# 04 · Reading the code: conventions, the component contract and normalized margins

Every class in `src/aircraft_closure/` follows the same small set of conventions. Once you know them you can read
any file quickly, predict where a given equation lives, and know which code is allowed to create variables. This notebook
covers them in the order you meet them when opening a file.

**In this notebook**
1. Names carry units.
2. Frozen dataclasses: immutable components, variants by `replace`, fields that hold numbers *or* variables.
3. The component contract: `get_mass()`, `get_limits()`, `evaluate(...)`, and nothing that solves or clips.
4. Interchangeable submodels and the "keep the simplest model" rule.
5. Normalized margins: `margin_below`, `margin_above`, `margin_report`, labels, and "binding".
6. Where the reasons live: the decision log, module docstrings, tests and sanity cases.

**Run time:** a few seconds.

In [1]:
from tutorial_helpers import *
import dataclasses, inspect, re
from dataclasses import replace

## 1. Names carry units

SI internally, and every dimensional name ends with its unit: `power_rated_W`, `speed_rad_s`, `torque_Nm`,
`area_disk_m2`, `voltage_bus_V`, `energy_capacity_J`, `mass_takeoff_kg`. Dimensionless names have no suffix
(`soc`, `efficiency`, `cl`, `advance_ratio`, `hybridization_electric`). Qualifiers come first: `max_`, `min_`, `rated_`,
`ref_`. So `max_torque_Nm` is a limit in N·m, `power_rated_W` a rating in W.

Let's measure how consistently this holds across all the component and vehicle dataclasses:

In [2]:
import importlib, pkgutil, aircraft_closure
unit_suffix = re.compile(r"_(W|kW|J|kg|m|m2|m3|s|Nm|N|V|A|ohm|rad_s|m_s|m_s2|Pa|K|C|deg|rad|As|kg_s|kg_m2|kg_m3|W_kg|J_kg|"
                         r"Nm_kg|W_s_rad|W_s3_rad3|W_Nm2|W_s2_rad2|ohm_J|ohm_m|A_m2|kg_A|J_K|K_W|W_K|J_kg_K|usd|usd_kg|usd_W|"
                         r"usd_J|usd_s|per_rad|per_deg|per_rev|Pa_s)$")
dimensionless_hints = ("ratio", "fraction", "factor", "efficiency", "count", "soc", "coefficient", "exponent", "cl",
                       "cd", "mach", "solidity", "kappa", "model", "label", "source", "mode", "polar", "increment",
                       "material", "conductor", "insulation", "discharge", "branches", "cell", "staircase", "shape",
                       "airfoil", "interference", "download", "trim", "blown", "smoothing", "effectiveness", "loading",
                       "thickness_to_chord", "taper", "aspect", "is_", "sign", "width", "k_e", "max_soc", "min_soc",
                       "advance", "softness", "size", "transition", "xtr", "n_crit", "lapse", "sources", "coefficients",
                       "topology", "locations", "cooling", "partial", "heat_exchanger", "component", "name", "symmetric")
fields_total, fields_named = 0, 0
unclear = []
for module_info in pkgutil.walk_packages(aircraft_closure.__path__, "aircraft_closure."):
    if ".export" in module_info.name or ".trajectory" in module_info.name:
        continue
    module = importlib.import_module(module_info.name)
    for _, cls in inspect.getmembers(module, inspect.isclass):
        if cls.__module__ != module.__name__ or not dataclasses.is_dataclass(cls):
            continue
        for f in dataclasses.fields(cls):
            fields_total += 1
            if unit_suffix.search(f.name) or any(h in f.name for h in dimensionless_hints):
                fields_named += 1
            else:
                unclear.append(f"{cls.__name__}.{f.name}")
print(f"{fields_named} of {fields_total} dataclass fields carry a unit suffix or are named as dimensionless")
print("the rest (read these with care):", ", ".join(unclear[:25]), "..." if len(unclear) > 25 else "")

733 of 848 dataclass fields carry a unit suffix or are named as dimensionless
the rest (read these with care): BuildupResult.aero, BuildupResult.breakdown, BuildupResult.oswald_span, InterferenceFactors.wing, InterferenceFactors.horizontal_tail, InterferenceFactors.vertical_tail, InterferenceFactors.fuselage, ScholzAerodynamics.thickness_location_chordwise, AeroResult.lift_to_drag, SimpleAerodynamics.thickness_location_chordwise, TailTrim.oswald_tail, LongitudinalResult.cm, LongitudinalStability.cm_ac_wing, Margin.value, MarginReportEntry.value, MarginReportEntry.compatible, PortSpec.domain, PortSpec.direction, Bus.domain, Connection.target, Connection.combine, Instance.ports, Residual.value, Mission.segments, MissionResult.segments ...


The rest are dimensionless quantities this crude word list does not know (`cm`, `lift_to_drag`, interference factors),
structural objects (ports, connections, segments, nested results) and margin values (always dimensionless). When you add code, follow the rule: it is the cheapest unit check there is.

## 2. Frozen dataclasses

Every component, model, requirement, condition and result is a `@dataclass(frozen=True)`. Three consequences:

In [3]:
from aircraft_closure.powertrain.components.motor import Motor, McDonaldMotorLossModel

motor = Motor(power_rated_W=200e3, max_torque_Nm=800.0)
try:
    motor.max_torque_Nm = 900.0
except dataclasses.FrozenInstanceError as error:
    print("1. immutable ->", type(error).__name__)

bigger = replace(motor, max_torque_Nm=900.0)
print("2. variants by replace() ->", bigger.max_torque_Nm, "(original still", motor.max_torque_Nm, ")")

opti = asb.Opti()
torque_peak_Nm = opti.variable(init_guess=800.0, scale=100.0)
sized = replace(motor, max_torque_Nm=torque_peak_Nm, power_rated_W=250.0 * torque_peak_Nm)
print("3. the same class holds variables ->", type(sized.max_torque_Nm).__name__, type(sized.get_mass()).__name__)

1. immutable -> FrozenInstanceError
2. variants by replace() -> 900.0 (original still 800.0 )
3. the same class holds variables -> MX MX


1. **Immutable.** A component cannot change behind your back. One motor object is shared by every flight point of a
   problem; a flight point that mutated it would corrupt all the others. Immutability rules that out by construction.
2. **Variants by `replace`.** A failure state, a pack with one string isolated, an assumption change for a study: all are
   `replace(original, field=new_value)`. You will see `replace` hundreds of times in `examples/halo_sizing.py`.
3. **Fields are typed `Any` on purpose.** The same field holds a float (a fixed design, a unit test) or an Opti
   variable / expression (being sized). **What is a design variable is decided by whoever constructs the object, not by
   the class.** The motor class never knows whether its torque rating is being sized.

## 3. The component contract

A powertrain component has exactly three methods (`docs/CODING_CONVENTIONS.md`, "Powertrain"):

| Method | Returns | Depends on |
|---|---|---|
| `get_mass()` | mass in kg (an expression if the ratings are variables) | the component's own fields |
| `get_limits()` | a small `...Limits` dataclass (max torque, max speed, voltage window, rated power...) | the component's own fields (turboshaft: also the atmosphere) |
| `evaluate(...)` | a named `...Result` dataclass for the operating inputs (speed, torque, voltage, current, thrust...) | operating inputs |

and **never**: creates an Opti variable, adds a constraint, iterates, clips, or reads a file. Look at the whole motor:

In [4]:
print(inspect.getsource(Motor))

@dataclass(frozen=True)
class Motor:
    power_rated_W: Any = 100000.0
    specific_power_W_kg: float = 5000.0
    max_speed_rad_s: Any = 1000.0
    max_torque_Nm: Any = 500.0
    min_voltage_V: Any = 400.0
    max_voltage_V: Any = 900.0
    loss_model: Any = field(default_factory=McDonaldMotorLossModel)
    mass_model: Any = None
    # Tier 19: LumpedThermalModel or None. With one, the power rating is a continuous (thermal) rating that
    # short peaks may exceed; callers then constrain temperature instead of rated power (plan 028).
    thermal_model: Any = None

    def get_mass(self):
        if self.mass_model is not None:
            return self.mass_model.mass_kg(self)
        return self.power_rated_W / self.specific_power_W_kg

    def get_limits(self):
        return MotorLimits(self.power_rated_W, self.max_speed_rad_s, self.max_torque_Nm,
                           self.min_voltage_V, self.max_voltage_V)

    def evaluate(self, speed_rad_s, torque_Nm, voltage_V):
        pow

Notice what `evaluate` does *not* do. Ask the motor for 3× its torque limit and it answers without complaint. The limit
is a separate piece of information for the caller to enforce:

In [5]:
motor = Motor(power_rated_W=200e3, max_torque_Nm=800.0, max_speed_rad_s=600.0)
state = motor.evaluate(speed_rad_s=300.0, torque_Nm=2400.0, voltage_V=700.0)
limits = motor.get_limits()
print(state)
print(limits)
print(f"torque used / allowed = {2400 / limits.max_torque_Nm:.1f} (the component neither clips nor complains)")
check("electrical power = shaft power + losses exactly", abs(state.power_electric_W - (300 * 2400 + state.power_loss_W)) < 1e-6)

MotorResult(power_shaft_W=720000.0, power_electric_W=961000.434027778, current_A=1372.8577628968258, power_loss_W=241000.434027778)
MotorLimits(power_rated_W=200000.0, max_speed_rad_s=600.0, max_torque_Nm=800.0, min_voltage_V=400.0, max_voltage_V=900.0)
torque used / allowed = 3.0 (the component neither clips nor complains)
PASS  electrical power = shaft power + losses exactly


Why so strict? Because in an all-at-once problem **only the caller can see the whole problem.** If the motor clipped torque
at its limit, the optimizer would see a flat, non-differentiable function and never learn that a bigger motor helps. If
it threw an error, an infeasible intermediate iterate (normal in all-at-once, tutorial 03) would crash the solve.
Returning the physics and the limits separately lets the caller turn the limit into a constraint, with a multiplier, by name.

**Where variables do get created.** Only in orchestration code: `performance/flight_point.py` (per-point states),
`mission/mission.py` (nothing: it chains points), and the drivers in `examples/` (design variables). Check with a search:

In [6]:
creators = {}
for path in sorted((repo_root / "src" / "aircraft_closure").rglob("*.py")):
    count = len(re.findall(r"opti\.variable\(", path.read_text(encoding="utf-8")))
    if count:
        creators[str(path.relative_to(repo_root / "src" / "aircraft_closure"))] = count
for name, count in creators.items():
    print(f"{count:3d} opti.variable(...) in {name}")
check("no powertrain component, vehicle or aerodynamics model creates a variable",
      not any(k.startswith(("powertrain" + "\\components", "powertrain/components", "vehicle", "aerodynamics"))
              for k in creators))

  8 opti.variable(...) in performance\flight_point.py
  7 opti.variable(...) in powertrain\cells.py
  5 opti.variable(...) in trajectory\corridor.py
 15 opti.variable(...) in trajectory\tiltrotor.py
PASS  no powertrain component, vehicle or aerodynamics model creates a variable


(`powertrain/cells.py` fits the battery resistance model to data with its own small `asb.Opti`: a data-layer fit, run
once, not part of any sizing. `trajectory/` holds the post-sizing trajectory problems.)

## 4. Interchangeable submodels, and keeping the simplest one

When fidelity changes but the operating inputs do not, the code uses a **submodel field** that any object with the right
method can fill (duck typing, no base classes):

| Owner | Field | Options (simplest first) |
|---|---|---|
| `Motor`, `Generator` | `loss_model` | `SimpleMotorLossModel`, `McDonaldMotorLossModel` |
| `Motor`, `Generator` | `mass_model` | `None` (power / specific power), `TorqueDensityMassModel`, `DatabaseMassModel`, `UnitMachineMassModel` |
| `Motor`, `Generator`, batteries | `thermal_model` | `None`, `LumpedThermalModel` |
| `SimpleTurboshaft` | `part_power_model` | `None`, `GeissPartPowerModel`, `CubicPartPowerModel`, `TabulatedPartPowerModel` |
| `SimpleTurboshaft` | `lapse_model` | `None` (σⁿ), `DensityTemperatureLapse` |
| `Wing` | `mass_model` | `None` (Raymer), `TiltrotorWingMassModel` |

When the *inputs* change, a **different class** with the same contract replaces the old one: `ActuatorDiskPropulsor` (no
rotor-speed dependence) vs `MomentumProfileRotor` (needs rotor speed); `Battery` (constant voltage) vs
`EquivalentCircuitBattery` (state of charge, RC history, temperature); `SimpleAerodynamics` / `ScholzAerodynamics` /
`BuildupAerodynamics`. The caller's code does not change.

**Rule 10: keep the simplest useful model.** Every higher-fidelity model sits next to the one it replaced, both stay
switchable and tested, so the effect of each model on the answer is traceable. That is why the Halo driver can still
reproduce every earlier reference design (tutorial 00 used one).

Swapping a submodel, concretely:

In [7]:
from aircraft_closure.powertrain.components.motor import SimpleMotorLossModel, TorqueDensityMassModel
simple = Motor(loss_model=SimpleMotorLossModel())
mcdonald = Motor(loss_model=McDonaldMotorLossModel(speed_peak_efficiency_rad_s=400.0, torque_peak_efficiency_Nm=200.0))
for label, m in (("simple loss model  ", simple), ("McDonald loss model", mcdonald)):
    r = m.evaluate(400.0, 200.0, 800.0)
    print(f"{label}: efficiency at 400 rad/s, 200 N m = {r.power_shaft_W / r.power_electric_W:.4f}")
print(f"mass, power / specific power: {Motor().get_mass():.1f} kg;  "
      f"torque density: {replace(Motor(), mass_model=TorqueDensityMassModel()).get_mass():.1f} kg")

simple loss model  : efficiency at 400 rad/s, 200 N m = 0.9881
McDonald loss model: efficiency at 400 rad/s, 200 N m = 0.9600
mass, power / specific power: 20.0 kg;  torque density: 33.3 kg


## 5. Normalized margins

`core/margins.py` is 44 lines and used for every limit in the repository:

In [8]:
from aircraft_closure.core import margins
print(inspect.getsource(margins))

"""Normalized margins: >= 0 is compatible, 0.1 means 10 % headroom.

Margins report; they never clip, resize or solve. Callers either constrain
`margin.value >= 0` in their own `asb.Opti` or report margins after a solve.
"""
from dataclasses import dataclass
from typing import Any


@dataclass(frozen=True)
class Margin:
    label: str
    value: Any


@dataclass(frozen=True)
class MarginReportEntry:
    label: str
    value: Any
    compatible: bool


def margin_below(label, value, limit):
    """Margin for `value <= limit`, normalized by the (positive) limit."""
    return Margin(label, (limit - value) / limit)


def margin_above(label, value, limit):
    """Margin for `value >= limit`, normalized by the (positive) limit."""
    return Margin(label, (value - limit) / limit)


def margin_report(margins, value_of=lambda value: value):
    """Numeric margins sorted from most to least critical.

    `value_of` maps a margin expression to a number, e.g. `solution.value`
    after an Opti s

- `margin_below(label, value, limit)` = (limit − value) / limit, for value ≤ limit.
- `margin_above(label, value, limit)` = (value − limit) / limit, for value ≥ limit.
- ≥ 0 is satisfied; 0.1 is 10 % headroom; −0.05 is 5 % over.
- The caller constrains `margin.value >= 0`; after the solve, `margin_report(margins, solution.value)` sorts them most
  critical first, and the drivers call a margin **binding** when |value| < 1e-5.

Normalizing by the limit does three jobs: IPOPT sees constraints of order one whatever the physics (a 10 kW and a 0.01 m
violation look alike); multipliers become comparable across constraints (tutorial 02); and a person reads "0.03" as "3 %
spare" without knowing the units. Labels carry the flight condition, the instance and the quantity with its unit, so a
binding list reads like `engine-out hover: battery discharge_power_W`.

A complete small example: constrain a motor to a duty point with margins, size its torque rating, and report:

In [9]:
from aircraft_closure.core.margins import margin_below, margin_above, margin_report
opti = asb.Opti()
torque_rating_Nm = opti.variable(init_guess=500.0, scale=100.0, lower_bound=10.0)
motor = Motor(max_torque_Nm=torque_rating_Nm, power_rated_W=200e3, max_speed_rad_s=600.0,
              mass_model=TorqueDensityMassModel())
duty = motor.evaluate(speed_rad_s=250.0, torque_Nm=700.0, voltage_V=750.0)
limits = motor.get_limits()
duty_margins = (
    margin_below("climb: motor torque_Nm", 700.0, limits.max_torque_Nm),
    margin_below("climb: motor speed_rad_s", 250.0, limits.max_speed_rad_s),
    margin_below("climb: motor power_shaft_W", duty.power_shaft_W, limits.power_rated_W),
    margin_above("climb: motor min_voltage_V", 750.0, limits.min_voltage_V),
    margin_below("climb: motor max_voltage_V", 750.0, limits.max_voltage_V),
)
opti.subject_to([m.value >= 0 for m in duty_margins])
opti.minimize(motor.get_mass())
solution = opti.solve(verbose=False)
for entry in margin_report(duty_margins, solution.value):
    print(f"{entry.value:+.4f}  {'BINDING ' if abs(entry.value) < 1e-5 else '        '}{entry.label}")
print(f"torque rating {solution.value(torque_rating_Nm):.1f} N m, mass {solution.value(motor.get_mass()):.1f} kg")
check("the lightest motor sits exactly on its torque limit", abs(solution.value(torque_rating_Nm) - 700.0) < 1e-3)

-0.0000  BINDING climb: motor torque_Nm


+0.1250          climb: motor power_shaft_W
+0.1667          climb: motor max_voltage_V
+0.5833          climb: motor speed_rad_s
+0.8750          climb: motor min_voltage_V
torque rating 700.0 N m, mass 46.7 kg
PASS  the lightest motor sits exactly on its torque limit


Note the limit here is a *variable* (the torque rating), so the margin (700 − T_max)/T_max is a ratio of expressions. That
is normal: margins are expressions like everything else.

Margins **report; they never resize.** Nothing in the code reads a negative margin and grows a part; only the optimizer
moves variables. `margin_report` refuses to compare a symbolic margin without `value_of`, so a margin cannot be
silently coerced to a number before the solve:

In [10]:
try:
    margin_report(duty_margins)
except Exception as error:
    print(type(error).__name__, "->", str(error).splitlines()[0][:100])

RuntimeError -> .../casadi/core/mx_node.cpp:205: Can only determine truth value of a numeric MX.


## 6. Where the reasons live

- **Module docstrings** state the equations, their sources and validity. Every module in `src/` opens with one; read it
  before the code.
- **Field comments** state each default's source (`# XV-15`, `# assumed`, `# Paudel et al. 2025 Fig. 8`).
- **`docs/decisions/NNN-*.md`**: one file per modelling step ("plan"), with what was tried, what was rejected and the
  results. Code comments cite them as "plan 033".
- **Module sanity cases**: most component modules end with `if __name__ == "__main__":` printing one evaluation.
- **Tests** mirror `src/` (`tests/powertrain/test_components.py` ...) and check identities, limiting cases and trends;
  `tests/integration/` reproduces documented results. Run one file with
  `uv run python -m unittest tests.powertrain.test_components`.

In [11]:
decisions = sorted((repo_root / "docs" / "decisions").glob("*.md"))
print(f"{len(decisions)} decision records, e.g.:")
for path in decisions[::6]:
    first = path.read_text(encoding="utf-8").splitlines()[0].lstrip("# ")
    print(f"   {path.name:<46} {first[:70]}")
cited = re.findall(r"plan (\d{3})", (repo_root / "examples" / "halo_sizing.py").read_text(encoding="utf-8"))
print(f"\nexamples/halo_sizing.py cites {len(set(cited))} different plans, {len(cited)} times")

39 decision records, e.g.:
   001-foundation-and-simple-powertrain.md        Foundation and simple powertrain
   007-stability-trim-tail-sizing.md              Conventional-tail stability, trim and failed-propulsor yaw control
   013-halo-class-sizing.md                       Halo-class two-rotor series-hybrid sizing
   019-trajectory-optimization.md                 Trajectory optimization on a sized aircraft (Tier 14)
   025-aerodynamics-buildup.md                    Aerodynamics build-up: AeroBuildup, Scholz hand check, blown wing, dow
   031-tiltrotor-geometry-openvsp.md              Tiltrotor geometry layout and OpenVSP export (Tier 23, plan 031)
   037-halo-reference-layout.md                   Halo reference from the drawn layout (plan 037; plan 032 on its branch

examples/halo_sizing.py cites 20 different plans, 81 times


## Summary

- Units in names; SI inside; convert at the boundary.
- Frozen dataclasses; variants by `replace`; fields hold numbers or variables, and the constructor decides which.
- Components: `get_mass`, `get_limits`, `evaluate`, nothing else; they never create variables, constrain, iterate or clip.
- Fidelity grows by submodel fields or by new classes behind the same contract; the simplest model stays.
- Every limit is a normalized margin with a descriptive label; the caller constrains it, the report names what binds.

## Check yourself

<details><summary>1. You want the motor to derate its torque above 120 °C. Where does that code go?</summary>

The physics (temperature as a function of losses and time) goes in a thermal submodel (`LumpedThermalModel` exists). The derating is a *limit*: the motor (or the thermal discipline) returns the derated maximum torque as an expression of temperature, and the caller adds a margin `torque <= derated max`. Nothing clips.
</details>

<details><summary>2. Why is <code>max_torque_Nm</code> typed <code>Any</code> rather than <code>float</code>?</summary>

So the same class can carry a fixed number (analysis, tests) or an Opti variable (sizing). Which one it is is the constructor's decision.
</details>

<details><summary>3. A margin label reads <code>bus-out hover 2/3: motor torque_Nm</code>. Decode it.</summary>

In the second of three sub-points of the bus-out failure hover, the motor's torque relative to its maximum torque. "2/3" comes from `subsegments` splitting a segment into points.
</details>

In [12]:
check_summary()

3 of 3 checks passed
